In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from photutils.aperture import CircularAperture, aperture_photometry
from astropy.stats import sigma_clip

In [ ]:
# reset defalult plotting values
plt.rcParams['figure.figsize'] = (10, 5)
plt.rc('font', family='sans-serif')
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')

# ASTR 350: Astronomical Techniques
# Lecture 19


### Prof. Robert Quimby <br> March 24, 2026

&copy; 2026 Robert Quimby

## Quiz 14 review

In [ ]:
# load the image data
image = fits.getdata('media/example.fits')

# reference star position
xref, yref = 569.42, 424.33

# supernova position
xsn, ysn = 544.94, 519.27

# refernece star magnitude
refmag = 15.000

In [ ]:
# local background pixels near the supernova and star
bg_sample = sigma_clip(image[400:600, 400:600])
bg_sample.mean()

In [ ]:
# show the positions of the SN and reference star
plt.figure(figsize=(7,7))
plt.imshow(image, origin='lower', cmap='gray', vmin=1400, vmax=1600);
plt.plot(xsn, ysn, marker='o', mfc='none', c='r', ms=35)
plt.plot(xref, yref, marker='o', mfc='none', c='y', ms=35)
plt.xlim(400, 600), plt.ylim(400, 600);

## Estimate the angular diameter of a star (in pixels)

In [ ]:
import astropy.units as u
pixel_scale = (0.358 * u.arcsecond / u.pixel)

# radius of the Sun
radius = ????

# distance to a (near by) star
distance = ????

# angular size
angle = ????

# angular size (in pixels)
(angle / pixel_scale).to(u.pixel)

## How big should our apertures be?

In [ ]:
# bright star
x1, y1 = 676.5416, 491.37543

# faint, isolated star
x2, y2 = 684.725, 356.464

In [ ]:
# show the positions of two other stars
plt.figure(figsize=(7,7))
plt.imshow(image, origin='lower', cmap='gray', vmin=1400, vmax=1600);
plt.plot(x1, y1, marker='o', mfc='none', c='b', ms=35)
plt.plot(x2, y2, marker='o', mfc='none', c='g', ms=35)
plt.xlim(400, 700), plt.ylim(300, 600);

In [ ]:
def get_profile(image, x, y, margin=15, axis=0):
    i, j = int(x), int(y)
    stamp = image[j - margin : j + margin + 1, i - margin : i + margin + 1]
    
    profile = stamp.sum(axis=axis)
    profile -= profile.min()
    profile /= profile.max()
    
    if axis == 0:
        delta = i - x
    else:
        delta = j - y
    offset = np.arange(profile.size) + delta - margin

    return offset, profile

In [ ]:
# get the profiles for each star
axis1, profile1 = get_profile(image, xref, yref)
axis2, profile2 = get_profile(image, x2, y2)

In [ ]:
plt.plot(axis1, profile1)
plt.plot(axis2, profile2)
plt.grid();

In [ ]:
# radius for our photometry aperture
radius = ????

## Add up the counts

In [ ]:
# counts from the reference star
aperture = CircularAperture( ???? )
phot_table = aperture_photometry( ???? )
ref_phot = phot_table[0]
refcounts = ref_phot['aperture_sum']
refcounts

In [ ]:
# counts from the supernova
aperture = CircularAperture( ???? )
phot_table = aperture_photometry( ???? )
sn_phot = phot_table[0]
counts = sn_phot['aperture_sum']
counts

In [ ]:
# uncertainty in the supernova counts
obj_var = counts
bg_var = aperture.area * bg_sample.var()
avbg_var = bg_sample.var() / bg_sample.size * aperture.area**2
ecounts = np.sqrt( obj_var + bg_var +  avbg_var)

## Convert from counts to magnitude

In [ ]:
mag = ????
emag = ????
print(f"supernova magnitude is {mag:.3f} +/- {emag:.3f}")

## Light Curves

* (a plot of) an object's brightness over time

In [ ]:
data = np.genfromtxt('media/sn_lc.dat', names='jd, mag, emag')
plt.errorbar(data['jd'], data['mag'], data['emag'], marker='o', ls='none')
plt.ylabel('Magnitude')
plt.xlabel('Julian Date')
plt.gca().invert_yaxis();

In [ ]:
# variable star
lc = np.genfromtxt('media/varstar.dat', names='mjd, mag, emag, flag')
lc.sort(order='mjd')
plt.plot(lc['mjd'], lc['mag'], 'ro');
plt.xlabel('Modified Julian Date')
plt.ylabel('Magnitude')
plt.gca().invert_yaxis()
plt.grid();
plt.xlim(57420, 57421.5)

## Scipy

Tutorial
- [notebook](../tutorials/scipy.ipynb)
- [video](https://youtu.be/6OEzjtuUMSs) (25 min)

## `fmin` example: fitting a linear relation

In [ ]:
# load the data
data = np.genfromtxt('media/xy.dat', names='x,y')
plt.plot(data['x'], data['y'], 'ro')
plt.grid();

In [ ]:
# define the model (y = mx + b)
def line_model(params, x):
    y = ????
    return y

In [ ]:
def sum_residuals_squared(params, data):
    model = line_model(params, data['x'])
    dY = data['y'] - model
    return ????

In [ ]:
# use scipy optimize
from scipy.optimize import fmin

????

In [ ]:
# plot the data and model
modelx = np.linspace(0, 2, 100)
modely = line_model( ???? )
plt.plot(data['x'], data['y'], 'ro')
plt.plot(modelx, modely)
plt.grid();

## When does Mars appear closest to Jupiter in the sky this year?

In [ ]:
from astropy.time import Time
from astropy.coordinates import get_body

# current Mars-Jupiter separation
????

In [ ]:
# plot the separation over time
times = ????
????

plt.scatter(times.datetime, sep)
plt.grid()

In [ ]:
from scipy.optimize import fmin
def mars_jup_sep(jd):
    time = Time(jd, format='jd')
    ????
    
jdmin = ????

## Example using `minimize`

In [ ]:
from scipy.optimize import minimize

def mars_jup_sep(jd):
    time = Time(jd, format='jd')
    ????
    
# initial guess
guess = ????

# set boundaries
bounds = ????

# call minimize
res = minimize(mars_jup_sep, guess, bounds=bounds)
jdmin = ????
Time(jdmin, format='jd').iso  , mars_jup_sep(jdmin)

In [ ]:
res = minimize(
    mars_jup_sep,
    guess,
    method=????,
    bounds=bounds,
    options=????
)
jdmin = ????
Time(jdmin, format='jd').iso  , mars_jup_sep(jdmin)

- Documentation for SciPy's [minimize function](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
- Explanation of [Nelder-Mead method](https://en.wikipedia.org/wiki/Nelder%E2%80%93Mead_method)
- Explanation of [Powel's method](https://en.wikipedia.org/wiki/Powell%27s_method)
- Explanation of popular [BFGS method](https://en.wikipedia.org/wiki/Broyden%E2%80%93Fletcher%E2%80%93Goldfarb%E2%80%93Shanno_algorithm)

## Are you getting this?

Do not download until instructed to do so:
- https://classroom.github.com/a/XUqbcoPS